# 1F1B
- （One Forward pass followed by One Backward pass）是分布式大模型训练中，流水线并行（Pipeline Parallelism）的一种核心调度策略。
- 它的核心思想是：<font color='red'>每完成一个微批次（Micro-batch）的前向传播，就尽可能早地安排它的反向传播，</font>即“一次前向紧跟一次反向”。
  - 这种“即时生产，即时消费”的模式，旨在解决传统流水线并行中显存占用过高的问题。

为了让你更透彻地理解，我们可以从它的诞生背景、运行机制和核心优势三个方面来拆解：
##  诞生背景：为了解决 GPipe 的“显存爆炸”
在 1F1B 之前，业界常使用 GPipe 策略。GPipe 采用的是“<font color='red'>先全部前向，再全部反向（F-then-B）</font>”的模式。
- GPipe 的痛点：<font color='red'>它要求 GPU 必须把所有微批次的前向计算结果（中间激活值）全部保存在显存里，直到最后一个微批次完成反向传播才能释放。</font>这就像快递分拣中心，必须把所有包裹都堆在仓库里才能开始统一派送，极易导致显存被撑爆。

##  运行机制：1F1B 的三个阶段
1F1B 通过让前向和反向计算交替进行，大幅缩短了激活值在显存中的存活时间。它的调度过程非常有序，通常分为三个阶段（以4张GPU、8个微批次为例）：
### 第一阶段：热身（Warm-up）
- 目标是“灌满”流水线。从 GPU0 开始，依次进行微批次的前向传播并向后传递。在这个阶段，只有前向计算在流动，反向传播还未开始。就像向一个空的水管里注水，直到水充满整个管道。
### 第二阶段：稳定状态（Steady State）
- 这是 1F1B 发挥威力的核心阶段。当流水线被灌满后，GPU 开始交替执行前向传播和反向传播（即 1F1B）。此时，系统达到了“<font color='red'>内存中每多一份前向激活数据，就释放一份前向激活数据</font>”的动态平衡。
### 第三阶段：收尾/冷却（Cool-down / Ending）
- 当所有微批次的前向计算都完成后，系统会顺次完成剩余微批次的反向传播，直到流水线排空。

##  核心优势：显存与计算的“双赢”
- 大幅降低峰值显存：在 1F1B 模式下，GPU 只要算完一个微批次的梯度，就可以立刻丢弃对应的前向激活值，释放出的显存马上可以被后续的微批次复用。研究表明，相比于 GPipe，1F1B 的峰值显存占用可降低约 37.5%。
- 保持高计算利用率：它既避免了“朴素流水线”中 GPU 大量空转等待的“流水线气泡”问题，又解决了 GPipe 的显存瓶颈，让 GPU 能够持续忙碌地交替计算。
## 总结来说：
- 1F1B 就像是一个智能的“潮汐车道”系统，它打破了“先集中生产，再统一消费”的僵局，通过让前向和反向计算紧凑地交替进行，在提升 GPU 计算利用率的同时，极大地缓解了显存压力。

## 🎯 场景设定
- 流水线级数：4 张 GPU（GPU 0, 1, 2, 3）。
- 微批次数量：8 个微批次（编号为 0 到 7）。
- 操作：F 代表前向传播（Forward），B 代表反向传播（Backward）。

### 1. 第一阶段：热身（Warm-up）
目标：把流水线“灌满”，让数据开始流动。
  - 在这个阶段，GPU 只做前向传播（F），不做反向（B）。为了不让后面的 GPU 等太久，GPU 0 会连续发送几个微批次。
- 规则：热身阶段通常发送 N - P 个微批次（这里 8 - 4 = 4 个）。
- 执行：<font color='red'>GPU 0 连续把微批次 0, 1, 2, 3 往前推。</font>
- 结果：<font color='green'>当热身结束时，微批次 0 刚好到达 GPU 3，微批次 1 在 GPU 2，微批次 2 在 GPU 1，微批次 3 在 GPU 0。</font>
  - <font color='blue'>此时，整个流水线里塞满了 4 个正在做前向的微批次。</font>

### 2. 第二阶段：稳定状态（Steady State）—— <font color='blue'>1F1B 的核心！</font>
- 目标：交替执行，释放显存。
<font color='green'>流水线满了之后，系统进入“1F1B”模式。</font>

- 规则：GPU 每执行 1 次前向（F），紧接着就执行 1 次反向（B）。
- 执行过程（<font color='blue'>以 GPU 0 为例</font>）：
  - F4：GPU 0 处理微批次 4 的前向，并把它传给 GPU 1。
  - B0：紧接着，GPU 0 <font color='red'>收到来自 GPU 1 的梯度</font>，<font color='blue'>开始处理微批次 0 的反向传播。</font>
  - F5：处理微批次 5 的前向。
  - B1：处理微批次 1 的反向。
  - F6 ... B2 ...
- 显存魔法：
  - <font color='red'>注意看，当 GPU 0 算完 B0 时，微批次 0 的整个生命周期就结束了，它的中间激活值可以立刻被释放</font>！
  - <font color='blue'>紧接着算 F5 时，就可以复用刚刚释放的显存</font>。这就是为什么 1F1B 省显存的原因。

### 3. 第三阶段：收尾（Cool-down）
- 目标：排空流水线。
  - 当前向的微批次（F7）发出去之后，没有新的前向可以做了，GPU 们就开始专心处理剩下的反向传播。
- <font color='red'>执行：GPU 0 依次执行 B4, B5, B6, B7。直到所有微批次的梯度都算完，流水线彻底清空。</font>

📊 直观的时间轴视图
- 如果我们把时间轴画出来，你会看到一张非常规律的“阶梯图”。以 GPU 0 为例，它的时间线是这样的：

In [ ]:
时间轴 -> 
GPU 0: [F0] [F1] [F2] [F3] | [F4] [B0] | [F5] [B1] | [F6] [B2] | [F7] [B3] | [B4] [B5] [B6] [B7]
         \_________________/   \_______/   \_______/   \_______/   \_______/   \_________________/
          热身阶段 (Warm-up)    \_______/   \_______/   \_______/   \_______/     收尾阶段 (Cool-down)
                              稳定阶段 (1F1B 交替执行)

In [ ]:
时间轴 ->
GPU 0: [F0] [F1] [F2] [F3] | [F4] [B0] | [F5] [B1] | [F6] [B2] | [F7] [B3] | [B4] [B5] [B6] [B7]
GPU 1:     [F0] [F1] [F2] | [F3] [B0] | [F4] [B1] | [F5] [B2] | [F6] [B3] | [F7] [B4] [B5] [B6] [B7]
GPU 2:         [F0] [F1] | [F2] [B0] | [F3] [B1] | [F4] [B2] | [F5] [B3] | [F6] [B4] | [F7] [B5] [B6] [B7]
GPU 3:             [F0] | [F1] [B0] | [F2] [B1] | [F3] [B2] | [F4] [B3] | [F5] [B4] | [F6] [B5] | [F7] [B6] [B7]

## 🔍 逐卡行为拆解（以 GPU 0 和 GPU 3 为例）

### 1. GPU 0（流水线起点）
- 热身最长：它必须一口气把 F0、F1、F2、F3 全部算完并发送出去，把后面的 GPU 都“喂饱”，才能开始接收梯度。
- 收尾最长：当它把最后一个前向 F7 发出去后，它手里还积压了 B4、B5、B6、B7 需要算完。
状态：6步热身 + 4步稳定(1F1B) + 6步收尾。
### 2. GPU 3（流水线终点）
- 无热身：它是第一个收到数据并算出最终结果的 GPU。它一算完 F0，马上就能拿到损失值（Loss）并开始算 B0。
- 无收尾：它不需要把数据往后传，算完最后一个反向 B7，它的任务就彻底结束了。
- 状态：0步热身 + 8步稳定(1F1B) + 0步收尾。
### 3. GPU 1 和 GPU 2（中间节点）
- 它们处于中间状态，热身和收尾的时间随着它们位置的靠后而逐渐减少。比如 GPU 1 需要热身 3 步，收尾 4 步；GPU 2 需要热身 2 步，收尾 2 步。
## ⚙️ 它们是如何“交接”的？（通信与计算的重叠）
1F1B 的精髓在于无缝衔接。我们来看看在“稳定阶段”，GPU 0 和 GPU 1 是如何配合的：
1. GPU 0 执行 F4：GPU 0 算完微批次 4 的前向传播，立刻通过通信（isend）把激活值发给 GPU 1。
2. GPU 1 接收并执行 F4：GPU 1 收到数据，开始算 F4。
3. GPU 1 执行 B0：GPU 1 算完 F4 后，紧接着算 B0，并把 B0 的梯度反向发给 GPU 0。
4. GPU 0 接收并执行 B0：GPU 0 收到梯度，立刻开始算 B0，释放显存。

💡 总结这个例子的核心意义：
- 在这个例子中，如果不用 1F1B（比如用 GPipe），GPU 0 必须一口气把 F0 到 F7 全部算完，然后再一口气算 B7 到 B0。<font color='red'>这意味着在算 F4~F7 的时候，F0~F3 的激活值必须一直死死地占着显存。</font>
- 而用了 1F1B，GPU 0 在算完 F4 之后，马上就去算 B0，把 F0 的显存释放了。这种“算一个前向，就赶紧算一个反向来腾地方”的策略，就是 1F1B 的精髓。